In [72]:
from langgraph.graph import StateGraph, MessagesState,START, END,add_messages
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage
from typing import TypedDict,Annotated,List,Sequence
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import ServerlessSpec,Pinecone
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough,RunnableLambda
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from langgraph.prebuilt import ToolNode,tools_condition
from langfuse import get_client
from langfuse.langchain import CallbackHandler


In [73]:
load_dotenv()

True

In [54]:
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "practice-rag" 

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [55]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

In [56]:
class RAGState(TypedDict):
    messages:Annotated[List[BaseMessage],add_messages]

In [57]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [58]:
rag_prompt = PromptTemplate.from_template(
   template= """You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [59]:
file_path = r"D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

In [60]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [61]:
embedder = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",output_dimensionality=384)


In [62]:
vectorstore = PineconeVectorStore.from_documents(
    texts,
    embedder,
    index_name=index_name
)

In [63]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [64]:
@tool
def rag_tool(query: str) -> dict:
    '''A tool that takes a user query, retrieves relevant information from the PDF, and returns an answer based on the retrieved context.
    Use this tool when the user asks factual/conceptual questions that might be answered from the stored documents.'''
    
    # Retrieve similar chunks from the vectorstore
    docs = retriever.invoke(query)
    context = format_docs(docs)
    
    # Generate answer using the context
    rag_chain = RunnableParallel({
        "question": RunnablePassthrough(),
        "context": RunnableLambda(lambda x: context)
    })
    
    output = (rag_chain | rag_prompt | llm | StrOutputParser()).invoke(query)
    
    return {
        "query": query,
        "output": output
    }

In [65]:
tools=[rag_tool]
llm_with_tools=llm.bind_tools(tools)

In [66]:
def chat_rag(state:RAGState):
    response=llm_with_tools.invoke(state["messages"])
    return {"messages":[response]}

In [67]:
tool_node=ToolNode(tools)

In [68]:
workflow=StateGraph(RAGState)
workflow.add_node("chat_rag",chat_rag)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "chat_rag")
workflow.add_conditional_edges(
    "chat_rag",
    tools_condition,
    {"tools": "tools",END: END}

)
workflow.add_edge("tools", "chat_rag")      

app=workflow.compile()

In [ ]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [79]:
result=app.invoke(
    {"messages": [HumanMessage(content=("What is module1?"))]},
    config={"callbacks": [langfuse_handler]}
)

In [80]:
print(result)

{'messages': [HumanMessage(content='What is module1?', additional_kwargs={}, response_metadata={}, id='cce3e11b-cbac-4214-bae2-86133ad63871'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'y3at98gsc', 'function': {'arguments': '{"query":"module1"}', 'name': 'rag_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 298, 'total_tokens': 314, 'completion_time': 0.025124823, 'completion_tokens_details': None, 'prompt_time': 0.022973233, 'prompt_tokens_details': None, 'queue_time': 0.05577736, 'total_time': 0.048098056}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49e0-14bf-7a82-8688-e69f5b43c615-0', tool_calls=[{'name': 'rag_tool', 'args': {'query': 'module1'}, 'id': 'y3at98gsc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'outp

In [ ]:
{'messages': [HumanMessage(content='What is module1?', additional_kwargs={}, response_metadata={}, id='9bd43cfd-292e-4b08-a7f6-9376703ec40e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm9j3gkynt', 'function': {'arguments': '{"query":"module1"}', 'name': 'rag_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 298, 'total_tokens': 314, 'completion_time': 0.023515003, 'completion_tokens_details': None, 'prompt_time': 0.018157489, 'prompt_tokens_details': None, 'queue_time': 0.04519553, 'total_time': 0.041672492}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49d7-7510-72e3-874d-86752068f665-0', tool_calls=[{'name': 'rag_tool', 'args': {'query': 'module1'}, 'id': 'm9j3gkynt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 298, 'output_tokens': 16, 'total_tokens': 314}), ToolMessage(content='{"query": "module1", "output": "I don\'t know based on the provided context."}', name='rag_tool', id='1503c548-216a-40b6-9475-24b6887f1531', tool_call_id='m9j3gkynt'), AIMessage(content="It seems that the function 'rag_tool' was unable to find any information about 'module1'.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 346, 'total_tokens': 367, 'completion_time': 0.029753601, 'completion_tokens_details': None, 'prompt_time': 0.021939944, 'prompt_tokens_details': None, 'queue_time': 0.046733529, 'total_time': 0.051693545}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d49d7-7f99-7eb0-ace5-109f180c97a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 346, 'output_tokens': 21, 'total_tokens': 367})]}

In [81]:
print(result["messages"][-1].content)

It seems that the function 'rag_tool' was unable to find any information about 'module1'.
